# Blog-to-X Thread Converter

## 1. Project Overview

**Task:** Convert a blog article into a short X (Twitter) thread — a series of connected posts, each under 280 characters, that preserve the article's key ideas in a compelling social format.

**Approach:** We compare rule-based chunking (split by character limit) against LLM-based conversion (semantic rewriting), test different thread structures (numbered, narrative, listicle), and evaluate coherence.

**Stack:** `LangChain` + `ChatOllama` + `qwen3.5:9b` — no API keys required.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Enforce hard constraints** (280-char limit per tweet) |
| 2 | Compare **rule-based** vs **LLM-based** content chunking |
| 3 | Design threads with **hooks, flow, and CTAs** |
| 4 | Test different **thread structures** |
| 5 | Evaluate **information density** vs **readability** |

## 3. Problem Statement

Blog posts are 500–2,000+ words. X threads are 5–15 tweets, each ≤280 characters. The challenge: compress without losing the core message, and make each tweet independently interesting.

## 4. Why This Matters

- **Content repurposing** maximizes the value of long-form writing
- **Hard length constraints** teach precise LLM control
- **Thread structure** teaches narrative design for short-form media

## 5. Setup

In [ ]:
import os, json, re, time, warnings
from textwrap import dedent
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

LLM_MODEL = "qwen3.5:9b"
llm = ChatOllama(model=LLM_MODEL, temperature=0.7)

def clean(text):
    if "<think>" in text: text = text.split("</think>")[-1].strip()
    return text.strip()

def ask(system, user):
    return clean(llm.invoke([SystemMessage(content=system), HumanMessage(content=user)]).content)

CHAR_LIMIT = 280
print(f"LLM ready | Tweet limit: {CHAR_LIMIT} chars")

## 6. Sample Blog Articles

In [ ]:
BLOGS = {
    "sql": {
        "title": "Why Every Developer Should Learn SQL",
        "article": dedent("""SQL isn't just for data analysts. Every developer interacts with databases, and understanding SQL makes you fundamentally better at your job.

First, debugging becomes faster. When a user reports slow loading times, you can query the database directly instead of guessing. A developer who can write a quick EXPLAIN ANALYZE is worth their weight in gold.

Second, architecture decisions improve. Understanding how indexes work, how JOINs affect performance, and when to denormalize gives you the context to design better schemas. The worst database designs I've seen came from developers who treated the DB as a black box.

Third, code reviews get sharper. You can spot N+1 queries, missing indexes, and unnecessary data fetches in pull requests. These are the bugs that don't show up in unit tests but crush performance in production.

The learning curve is gentler than you think. Start with SELECT, WHERE, and JOIN — you'll handle 80% of real-world queries. Then learn GROUP BY, window functions, and CTEs for the remaining 20%. Indexing and query optimization come with practice.

Resources: SQLBolt for interactive basics, Use The Index Luke for optimization, and your own production database for real-world practice (with read-only access, obviously).

SQL has been around for 50 years because it works. Every framework changes, but your SQL skills will transfer to every job you ever have.""")
    },
    "code_reviews": {
        "title": "How to Give Code Reviews That Don't Make People Hate You",
        "article": dedent("""Code reviews are one of the highest-leverage activities in software engineering. They catch bugs, spread knowledge, and raise quality. But done poorly, they create resentment, slow teams down, and make junior developers feel terrible.

The first rule: review the code, not the person. Instead of "you did this wrong," say "this approach might cause issues because..." The difference is subtle but the impact on psychological safety is enormous.

Second, separate nitpicks from blockers. Formatting preferences, naming style, and minor refactors are not worth blocking a PR. Mark them as "nit:" so the author knows they're optional. Reserve "blocking:" for actual bugs, security issues, or design problems.

Third, explain the why. "Use a map instead of a loop" is less helpful than "Using a map here would be O(1) lookup instead of O(n) scanning, which matters because this runs on every API request." Context transforms criticism into teaching.

Fourth, praise good work. If someone wrote an elegant solution, say so. If the test coverage is thorough, acknowledge it. Positive feedback reinforces good practices more effectively than negative feedback corrects bad ones.

Fifth, timebox your reviews. If a PR takes more than 30 minutes to review, it's probably too large. Ask the author to break it up. Small, focused PRs lead to better reviews and faster iteration.

The goal of code review isn't perfection. It's continuous improvement with mutual respect.""")
    },
}

for key, blog in BLOGS.items():
    print(f"{key}: {blog['title']} | {len(blog['article'].split())} words")

---

## 7. Approach 1: Rule-Based Chunking

In [ ]:
def rule_based_thread(text, max_tweets=8):
    """Split article into tweet-sized chunks by sentence."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    tweets = []
    current = ""
    for sent in sentences:
        if len(current) + len(sent) + 1 < CHAR_LIMIT - 10:  # Leave room for numbering
            current = (current + " " + sent).strip() if current else sent
        else:
            if current:
                tweets.append(current.strip())
            current = sent if len(sent) < CHAR_LIMIT else sent[:CHAR_LIMIT - 3] + "..."
    if current:
        tweets.append(current.strip())

    # Format as thread
    thread = []
    for i, tweet in enumerate(tweets[:max_tweets]):
        prefix = "\ud83e\uddf5" if i == 0 else f"{i}."
        thread.append(f"{prefix} {tweet}")
    thread.append(f"\n{len(thread)+1}. Found this useful? Repost \u267b\ufe0f and follow for more!")
    return thread

blog = BLOGS["sql"]
thread = rule_based_thread(blog["article"])
print("RULE-BASED THREAD")
print("=" * 50)
for t in thread:
    over = len(t) > CHAR_LIMIT
    print(f"{'\u274c' if over else '\u2705'} [{len(t):3d}] {t}")
    print()

## 8. Approach 2: LLM-Based Conversion

In [ ]:
THREAD_SYSTEM = """Convert this blog post into an X (Twitter) thread.

Rules:
- Create 6-10 tweets
- EACH tweet must be under 280 characters (this is critical)
- First tweet: hook that makes people want to read the thread
- Number tweets (1/, 2/, etc.)
- Last tweet: CTA (follow, repost, or question)
- Each tweet should be independently interesting
- Use 1-2 emojis per tweet maximum
- Preserve the article's key insights

Return ONLY the thread, one tweet per line, separated by blank lines."""

def llm_thread(blog):
    resp = ask(THREAD_SYSTEM, f"Blog title: {blog['title']}\n\n{blog['article']}")
    tweets = [t.strip() for t in resp.split("\n") if t.strip() and len(t.strip()) > 10]
    return tweets

for key, blog in BLOGS.items():
    print(f"\n{'=' * 60}")
    print(f"LLM THREAD: {blog['title']}")
    print("=" * 60)
    tweets = llm_thread(blog)
    violations = 0
    for t in tweets:
        over = len(t) > CHAR_LIMIT
        if over: violations += 1
        print(f"{'\u274c' if over else '\u2705'} [{len(t):3d}] {t}")
        print()
    print(f"Tweets: {len(tweets)} | Violations: {violations}")

## 9. Thread Structure Variants

In [ ]:
structures = {
    "Numbered list": "Format as a numbered list of key points. Each tweet is one insight.",
    "Narrative story": "Tell it as a story. Build tension. Reveal the lesson at the end.",
    "Contrarian take": "Lead with a controversial opinion. Challenge assumptions. Back up with evidence.",
}

blog = BLOGS["code_reviews"]
for name, instruction in structures.items():
    sys = f"{THREAD_SYSTEM}\n\nStyle: {instruction}"
    tweets = [t.strip() for t in ask(sys, f"Blog:\n{blog['article']}").split("\n") if t.strip() and len(t.strip()) > 10]
    print(f"\n--- {name} ({len(tweets)} tweets) ---")
    for t in tweets[:3]:
        print(f"  [{len(t):3d}] {t}")
    if len(tweets) > 3:
        print(f"  ... +{len(tweets)-3} more tweets")

## 10. Quality Evaluation

In [ ]:
EVAL_SYSTEM = """Evaluate this X thread converted from a blog article.
Score 1-5 on:
1. hook: Does the first tweet stop the scroll? 
2. flow: Do tweets connect logically?
3. density: Key info preserved in compressed form?
4. standalone: Is each tweet interesting alone?
5. cta: Does the last tweet drive engagement?

Return JSON: {"hook": N, "flow": N, "density": N, "standalone": N, "cta": N, "total": N}"""

blog = BLOGS["sql"]
rule_tweets = rule_based_thread(blog["article"])
llm_tweets = llm_thread(blog)

for name, tweets in [("Rule-based", rule_tweets), ("LLM-based", llm_tweets)]:
    thread_text = "\n".join(tweets)
    resp = ask(EVAL_SYSTEM, f"Blog: {blog['title']}\n\nThread:\n{thread_text}")
    text = clean(resp)
    s, e = text.find("{"), text.rfind("}") + 1
    if s >= 0 and e > s:
        try:
            scores = json.loads(text[s:e])
            total = sum(scores.get(k, 0) for k in ["hook","flow","density","standalone","cta"])
            print(f"{name}: {total}/25 | hook={scores.get('hook','?')} flow={scores.get('flow','?')} density={scores.get('density','?')}")
        except:
            print(f"{name}: (parse error)")

## 11. Character Limit Enforcement

LLMs often exceed the 280-character limit. Let's add a post-processing step.

In [ ]:
SHORTEN_SYSTEM = f"""Rewrite this tweet to be under {CHAR_LIMIT} characters.
Keep the core message. Remove filler words. Be concise.
Return ONLY the shortened tweet."""

tweets = llm_thread(BLOGS["code_reviews"])
print("POST-PROCESSED THREAD")
print("=" * 50)
for t in tweets:
    if len(t) > CHAR_LIMIT:
        shortened = ask(SHORTEN_SYSTEM, f"Tweet ({len(t)} chars):\n{t}")
        ok = len(shortened) <= CHAR_LIMIT
        print(f"{'\u2705' if ok else '\u274c'} [{len(shortened):3d}] {shortened}")
    else:
        print(f"\u2705 [{len(t):3d}] {t}")
    print()

## 12. Common Mistakes & Key Takeaways

| Mistake | Fix |
|---------|-----|
| Not enforcing 280-char limit | Add post-processing shortening step |
| Weak opening tweet | Hook must stop the scroll — test multiple options |
| Tweets that don't stand alone | Each tweet should be quotable by itself |
| No CTA at the end | Last tweet should ask a question or request repost |

| # | Takeaway |
|---|----------|
| 1 | **Rule-based chunking** is fast but produces poor threads (sentences split awkwardly) |
| 2 | **LLM conversion** produces better threads but needs character-limit enforcement |
| 3 | **Thread structure** (list vs narrative vs contrarian) changes engagement patterns |
| 4 | **Post-processing** (shortening over-limit tweets) is essential for production use |
| 5 | **Each tweet should work alone** — people often quote individual tweets from threads |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*